# truck-n-trailer

<a href="https://colab.research.google.com/github/ericbreh/truck-n-trailer/blob/main/truck-n-trailer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import sys
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
  !pip install polytope
  !pip install -q pyomo
  !apt-get install -y -qq glpk-utils
  !apt-get install -y -qq coinor-cbc
  !wget -N -q "https://matematica.unipv.it/gualandi/solvers/ipopt-linux64.zip"
  !unzip -o -q ipopt-linux64

import numpy as np
import matplotlib.pyplot as plt
import pyomo.environ as pyo
from matplotlib.animation import FuncAnimation

### Core Dynamics

- states: `[x, y, θ_truck, θ_trailer]`
- controls: `[v, φ]` (velocity and steering angle)

In [ ]:
def truck_trailer_dynamics(q, u, L, d):
    x, y, theta_t, theta_l = q
    v, phi = u

    x_dot = v * np.cos(theta_t)
    y_dot = v * np.sin(theta_t)
    theta_t_dot = (v / L) * np.tan(phi)
    theta_l_dot = (v / d) * np.sin(theta_t - theta_l)

    return np.array([x_dot, y_dot, theta_t_dot, theta_l_dot])

### Pyomo Model

In [ ]:
def build_pyomo_model(params):
    N = params["N"]
    dt = params["dt"]
    L = params["L"]
    d = params["d"]
    target = params["target"]

    # Control bounds
    v_min = params.get("v_min", -3.0)
    v_max = params.get("v_max", 3.0)
    phi_min = params.get("phi_min", -1.5)
    phi_max = params.get("phi_max", 1.5)

    # Objective weights
    control_w = params.get("w_control", 0.01)

    model = pyo.ConcreteModel()

    # Sets
    model.T = pyo.RangeSet(0, N)       # states 0..N
    model.K = pyo.RangeSet(0, N - 1)   # controls 0..N-1
    model.S = pyo.RangeSet(0, 3)       # state components 0..3

    # State variables
    model.x = pyo.Var(model.S, model.T, domain=pyo.Reals)

    # Control variables
    model.v = pyo.Var(model.K, bounds=(v_min, v_max))
    model.phi = pyo.Var(model.K, bounds=(phi_min, phi_max))

    # Forward Euler discretization
    def dyn_rule(model, s, k):
        if s == 0:  # x_dot = v * cos(theta_t)
            return model.x[0, k+1] == model.x[0, k] + dt * (model.v[k] * pyo.cos(model.x[2, k]))
        if s == 1:  # y_dot = v * sin(theta_t)
            return model.x[1, k+1] == model.x[1, k] + dt * (model.v[k] * pyo.sin(model.x[2, k]))
        if s == 2:  # theta_t_dot = (v / L) * tan(phi)
            return model.x[2, k+1] == model.x[2, k] + dt * ((model.v[k] / L) * pyo.tan(model.phi[k]))
        if s == 3:  # theta_l_dot = (v / d) * sin(theta_t - theta_l)
            return model.x[3, k+1] == model.x[3, k] + dt * ((model.v[k] / d) * pyo.sin(model.x[2, k] - model.x[3, k]))

    model.dyn = pyo.Constraint(model.S, model.K, rule=dyn_rule)

    # Jackknife prevention constraint
    max_jackknife = params.get(
        "max_jackknife_angle", np.pi/2)

    def jackknife_rule(model, t):
        return pyo.inequality(-max_jackknife,
                              model.x[2, t] - model.x[3, t],
                              max_jackknife)
    model.jackknife_cons = pyo.Constraint(model.T, rule=jackknife_rule)

    # Objective: minimize final state error + control effort
    def obj_rule(model):
        final_err = sum((model.x[i, N] - target[i])**2 for i in range(4))
        control_effort = sum(model.v[k]**2 + model.phi[k]**2 for k in model.K)
        return final_err + control_w * control_effort

    model.obj = pyo.Objective(rule=obj_rule, sense=pyo.minimize)

    return model

### MPC

In [ ]:
def set_initial_guess(model, u_guess):
    N = u_guess.shape[1]
    for k in range(N):
        model.v[k].set_value(float(u_guess[0, k]))
        model.phi[k].set_value(float(u_guess[1, k]))


def extract_solution(model, params):
    N = params["N"]
    x_opt = np.zeros((4, N + 1))
    u_opt = np.zeros((2, N))
    
    for i in range(4):
        for t in range(N + 1):
            x_opt[i, t] = pyo.value(model.x[i, t])
    
    for k in range(N):
        u_opt[0, k] = pyo.value(model.v[k])
        u_opt[1, k] = pyo.value(model.phi[k])
    
    return x_opt, u_opt


def run_mpc_loop(params):
    MAX_STEPS = params.get("max_steps", 150)
    TARGET_TOL = params.get("target_tol", 0.5)
    
    # Build model once
    model = build_pyomo_model(params)
    
    # Configure solver
    solver = pyo.SolverFactory("ipopt")
    solver.options["max_iter"] = params.get("solver_max_iter", 500)
    solver.options["tol"] = params.get("solver_tol", 1e-6)
    solver.options["print_level"] = params.get("solver_print_level", 0)
    
    # Initialize
    q_current = params["q0"].copy()
    N = params["N"]
    
    # Initial control guess
    v_init = params.get("v_init", 0)
    u_guess = np.zeros((2, N))
    u_guess[0, :] = v_init
    
    # Fix initial state
    for i in range(4):
        model.x[i, 0].fix(float(q_current[i]))
    
    q_history = [q_current.copy()]
    u_history = []
    step = 0
    
    print(f"Starting MPC simulation...")
    
    while np.linalg.norm(q_current[:2] - params["target"][:2]) > TARGET_TOL and step < MAX_STEPS:
        # Update initial state constraint
        for i in range(4):
            model.x[i, 0].fix(float(q_current[i]))
        
        # Warm start
        set_initial_guess(model, u_guess)
        
        # Solve optimization
        result = solver.solve(model, tee=False)
        
        # Check convergence
        term = result.solver.termination_condition
        if term not in (pyo.TerminationCondition.optimal, pyo.TerminationCondition.locallyOptimal):
            print(f"Solver warning: {term} at step {step}")
            break
        
        # Extract solution
        x_opt, u_opt = extract_solution(model, params)
        
        # Simulate one step
        u_star = u_opt[:, 0]
        q_dot = truck_trailer_dynamics(q_current, u_star, params["L"], params["d"])
        q_next = q_current + params["dt"] * q_dot
        
        # Record
        q_history.append(q_next.copy())
        u_history.append(u_star.copy())
        
        # Update state
        q_current = q_next.copy()
        step += 1
        
        # Shift horizon for warm start
        u_guess = np.hstack([u_opt[:, 1:], u_opt[:, -1:]])
    
    print(f"MPC finished in {step} steps. Final error: {np.linalg.norm(q_current[:2] - params['target'][:2]):.3f} m")
    
    return np.array(q_history), np.array(u_history)

### Simulation

In [ ]:
def plot_trajectory(q_hist, u_hist, params):
    q_hist = np.atleast_2d(q_hist)
    if q_hist.shape[0] == 4:
        q_hist = q_hist.T
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    # Trajectory plot
    ax1.plot(q_hist[:, 0], q_hist[:, 1], "-o", markersize=3, label="Trajectory")
    ax1.plot(params["target"][0], params["target"][1], "rx", markersize=12, mew=2, label="Target")
    ax1.set_xlabel("x (m)")
    ax1.set_ylabel("y (m)")
    ax1.set_title("Truck-Trailer Trajectory")
    ax1.grid(True, alpha=0.3)
    ax1.axis("equal")
    ax1.legend()
    
    # Control plot
    if len(u_hist) > 0:
        t = np.arange(len(u_hist)) * params["dt"]
        ax2.step(t, u_hist[:, 0], where="post", label="v (m/s)", linewidth=2)
        ax2.step(t, u_hist[:, 1], where="post", label="φ (rad)", linewidth=2)
        ax2.set_xlabel("Time (s)")
        ax2.set_ylabel("Control")
        ax2.set_title("Applied Controls")
        ax2.grid(True, alpha=0.3)
        ax2.legend()
    
    plt.tight_layout()
    plt.show()


def animate_trajectory(q_hist, params, interval=80):
    q_hist = np.atleast_2d(q_hist)
    if q_hist.shape[0] == 4:
        q_hist = q_hist.T
    
    L = params["L"]
    d = params["d"]
    M = len(q_hist)
    
    fig, ax = plt.subplots(figsize=(9, 9))
    ax.set_aspect("equal")
    ax.grid(True, alpha=0.3)
    ax.set_xlabel("x (m)")
    ax.set_ylabel("y (m)")
    
    # Set axis limits
    all_x = np.concatenate([q_hist[:, 0], [params["target"][0]]])
    all_y = np.concatenate([q_hist[:, 1], [params["target"][1]]])
    margin = max(6.0, 0.2 * max(np.ptp(all_x), np.ptp(all_y)))
    ax.set_xlim(all_x.min() - margin, all_x.max() + margin)
    ax.set_ylim(all_y.min() - margin, all_y.max() + margin)
    
    # Plot full path
    ax.plot(q_hist[:, 0], q_hist[:, 1], "k--", alpha=0.2, linewidth=1, label="Path")
    ax.plot(params["target"][0], params["target"][1], "rx", ms=12, mew=2, label="Target")
    
    # Animation artists
    truck_line, = ax.plot([], [], "b-", lw=5, label="Truck", solid_capstyle="round")
    trailer_line, = ax.plot([], [], "r-", lw=5, label="Trailer", solid_capstyle="round")
    truck_pts, = ax.plot([], [], "bo", ms=8)
    trailer_pts, = ax.plot([], [], "ro", ms=8)
    hitch_pt, = ax.plot([], [], "ko", ms=6)
    title_text = ax.text(0.5, 1.02, "", ha="center", va="bottom", transform=ax.transAxes, fontsize=11)
    
    def init():
        truck_line.set_data([], [])
        trailer_line.set_data([], [])
        truck_pts.set_data([], [])
        trailer_pts.set_data([], [])
        hitch_pt.set_data([], [])
        return truck_line, trailer_line, truck_pts, trailer_pts, hitch_pt, title_text
    
    def update(frame):
        x, y, theta_t, theta_l = q_hist[frame]
        
        # Truck: from rear axle to front axle
        x_front = x + L * np.cos(theta_t)
        y_front = y + L * np.sin(theta_t)
        
        # Trailer: from hitch (x, y) to trailer axle
        x_trailer = x - d * np.cos(theta_l)
        y_trailer = y - d * np.sin(theta_l)
        
        truck_line.set_data([x, x_front], [y, y_front])
        trailer_line.set_data([x, x_trailer], [y, y_trailer])
        truck_pts.set_data([x, x_front], [y, y_front])
        trailer_pts.set_data([x, x_trailer], [y, y_trailer])
        hitch_pt.set_data([x], [y])
        title_text.set_text(f"Step {frame+1}/{M}")
        
        return truck_line, trailer_line, truck_pts, trailer_pts, hitch_pt, title_text
    
    anim = FuncAnimation(fig, update, init_func=init, frames=M, interval=interval, blit=True)
    ax.legend(loc="upper right")
    
    return anim

### Forward Point-to-Point Navigation

In [ ]:
params_forward_point = {
    # Physical parameters
    "L": 2.0,           # truck wheelbase (m)
    "d": 5.0,           # hitch-to-trailer distance (m)
    
    # Target state [x, y, theta_t, theta_l]
    "target": np.array([np.random.uniform(0, 15), np.random.uniform(-15,15), 0.0, 0.0]),
    
    # Initial state [x, y, theta_t, theta_l]
    "q0": np.array([0.0, 0.0, 0.0, 0.0]),
    
    # MPC parameters
    "dt": 0.1,          # time step (s)
    "N": 10,            # prediction horizon
    
    # Control bounds
    "v_min": 0.0,       # forward only
    "v_max": 3.0,
    "phi_min": -1.5,
    "phi_max": 1.5,

    # Constraints
    "max_jackknife_angle": np.pi/2,
    
    # Objective weights
    "w_control": 0.01,
    
    # Simulation parameters
    "max_steps": 200,
    "target_tol": 0.5,
    
    # Solver settings
    "solver_max_iter": 500,
    "solver_tol": 1e-6,
    "solver_print_level": 0,
    
    # Initial guess
    "v_init": 0.5,
}

# Run MPC
q_hist_fwd, u_hist_fwd = run_mpc_loop(params_forward_point)

# Visualize results
plot_trajectory(q_hist_fwd, u_hist_fwd, params_forward_point)

# Animate
anim_fwd = animate_trajectory(q_hist_fwd, params_forward_point, interval=80)

# Display animation
from IPython.display import HTML, display
display(HTML(anim_fwd.to_jshtml()))

### Reverse Point-to-Point Navigation

In [ ]:
params_reverse_point = {
    # Physical parameters
    "L": 2.0,           # truck wheelbase (m)
    "d": 5.0,           # hitch-to-trailer distance (m)
    
    # Target state [x, y, theta_t, theta_l]
    # "target": np.array([-10.0, -12.0, 0.0, 0.0]),
    "target": np.array([np.random.uniform(0, -15), np.random.uniform(-15, 15), 0.0, 0.0]),
    
    # Initial state [x, y, theta_t, theta_l]
    "q0": np.array([0.0, 0.0, 0.0, 0.0]),
    
    # MPC parameters
    "dt": 0.1,          # time step (s)
    "N": 30,            # prediction horizon
    
    # Control bounds
    "v_min": -3.0,      # reverse only
    "v_max": 0.0,
    "phi_min": -1.5,
    "phi_max": 1.5,

    # Constraints
    "max_jackknife_angle": np.pi/2,
    
    # Objective weights
    "w_control": 0.01,
    
    # Simulation parameters
    "max_steps": 200,
    "target_tol": 0.5,
    
    # Solver settings
    "solver_max_iter": 500,
    "solver_tol": 1e-6,
    "solver_print_level": 0,
    
    # Initial guess
    "v_init": -0.5,
}

# Run MPC
q_hist_rev, u_hist_rev = run_mpc_loop(params_reverse_point)

# Visualize results
plot_trajectory(q_hist_rev, u_hist_rev, params_reverse_point)

# Animate
anim_rev = animate_trajectory(q_hist_rev, params_reverse_point, interval=80)

# Display animation
from IPython.display import HTML, display
display(HTML(anim_rev.to_jshtml()))